### Introduction to positional embedding (without being trained on any parameters)
1. Collect the data and do data sampling using dataloaders and datasets.
2. Convert text into token and token id's using BPE algorithm.
3. Represent token id's into vector representation using vector embeddings (feature dimensions, vocab_size)
4. Now, represent this token embeddings to do positional embedding (absolute or relative) to make each token in position to each other.

In [5]:
#Importing raw dataset from text file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_data = f.read()

In [19]:
# Dataset class to have data in batches input-target pair
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class GPTDatasetsV1(Dataset):

    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        #Encoding the text with help of tokenizer
        tokens =  tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

        #Determining max_length, stride of each batch_size of text
        for i in range(0, len(tokens) - max_length, stride):
            input_chunks = tokens[i: i + max_length]
            target_chunks = tokens[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.input_ids)


# Dataloader function to load the datasets in batches of size and do parallel processing in batches for multiple GPU architectures

def data_loader_V1(text, batch_size= 8, max_length= 4, stride= 4, num_workers= 0, shuffle= False, drop_last= False):
    # Defining tokeizer to run the encode for text
    tokenizer = tiktoken.get_encoding("gpt2")

    # Preparing datasets to be loaded in batches
    load_dataset = GPTDatasetsV1(text, tokenizer, max_length, stride)

    # Running dataloder call with dataset values
    dataloader = DataLoader(
        load_dataset,
        batch_size= batch_size,
        shuffle= shuffle,
        num_workers= num_workers,
        drop_last= drop_last,
    )

    return dataloader

In [27]:
# Get the input-target pair of input_ids and target_ids in batches

data_loader = data_loader_V1(raw_data)
data_iter = iter(data_loader)
input, target = next(data_iter)

print(f'Input Token batch Matrix: \n {input}')
print(f'Target Token batch Matrix: \n {target}')

Input Token batch Matrix: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Target Token batch Matrix: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [30]:
# Step 3:- Converting these input tokens id matrix into vector embeddings
# Our input batch matrix has shape, 8 * 4 (batch_size * context_length)
# The vector embedding matrix for this batch will be 8 * 4 * 256 (batch_size, context length, number of feature per token id's)
vocab_size = 50257
feature_dims = 256

# Initializing Embedding matrix with random values
vector_embeddings_layer = torch.nn.Embedding(vocab_size, feature_dims)

# Converting input-target matrix into token embeddings (8, 4, 256)
token_embeddings = vector_embeddings_layer(input)

print(f'Token Embedding layer shape: {token_embeddings.shape}')

Token Embedding layer shape: torch.Size([8, 4, 256])


In [38]:
# Step 4:- Converting Token Embedding layer + adding positional embedding layer to get input embeddings to feed to transformer layer
context_window = max_length = 4

positional_embedding_layer = torch.nn.Embedding(context_window, feature_dims)

# Making positional embedding matrix with dim (4 * 256)
positional_embedding = positional_embedding_layer(torch.arange(context_window))
print(f'Positional Embedding Shape: {positional_embedding.shape}')


# Adding Token/Vector embeddings + Positional embeddings = input embeddings
input_embeddings = token_embeddings + positional_embedding

print(f'Without Optimized Input Embedding Matrix: \n {input_embeddings}')
print(f'Without Optimized Input Embedding Matrix Shape: \n {input_embeddings.shape}')

Positional Embedding Shape: torch.Size([4, 256])
Without Optimized Input Embedding Matrix: 
 tensor([[[-1.2897e+00,  4.0786e-01, -2.2517e-01,  ..., -8.6110e-01,
          -2.9774e+00,  2.1189e+00],
         [ 5.4181e-01, -1.1348e+00, -6.5016e-01,  ...,  1.0202e+00,
          -2.3403e+00, -2.9774e+00],
         [ 2.9037e-01, -8.1915e-01,  1.2173e+00,  ...,  4.3133e-01,
           8.4400e-01,  1.2070e+00],
         [-2.4182e+00,  2.4126e+00, -6.6130e-01,  ...,  4.8548e-01,
           2.7408e-01,  1.6995e+00]],

        [[ 3.1937e-01,  1.0401e+00,  7.6213e-01,  ...,  7.9023e-02,
          -1.1670e+00,  2.3717e+00],
         [ 1.4973e+00, -2.9218e+00, -5.3015e-01,  ..., -1.1140e+00,
          -9.4154e-01, -8.3178e-01],
         [ 9.8869e-01,  1.2273e+00,  7.0041e-01,  ...,  5.5065e-01,
          -1.1552e-01,  1.3038e+00],
         [-1.9118e+00,  8.9823e-01, -2.6748e-01,  ..., -5.9972e-02,
           1.5042e+00,  8.4134e-01]],

        [[ 5.6641e-01,  4.1401e-01,  1.7730e+00,  ..., -4.5672e